# Multi-Family Biophysical Featurization: Beyond Pairwise Contacts

[![Open in JupyterLab](https://img.shields.io/badge/Launch-JupyterLab-orange.svg)](https://jupyter.org)
[![Python 3.10+](https://img.shields.io/badge/python-3.10+-blue.svg)](https://www.python.org/)

Most molecular dynamics analysis packages limit featurization to pairwise contact maps. MolScope provides a rich library of **12+ biophysical feature families**, allowing researchers to track secondary structure transitions, rotamer flips, solvent exposure, and electrostatic networks.

In this notebook, we explore the feature plugins available in MolScope:
1. **Cα Distances & Interaction Graphs**
2. **Secondary Structure Tracking via DSSP**
3. **Backbone & Sidechain Dihedrals ($\phi, \psi, \chi_1$)**
4. **Solvent Accessible Surface Area (SASA)**
5. **Salt Bridges & Electrostatic Networks**
6. **Aromatic $\pi$-$\pi$ Stacking Geometries**


## 1. Feature Family Overview


In [ ]:
import pandas as pd
from molscope.registries import get_feature_registry

registry = get_feature_registry()
print(f"Registered Feature Plugins in MolScope: {len(registry.list_all())}")
for name in registry.list_all():
    print(f"  - {name}")


## 2. Feature Configurations and Mathematical Definitions

| Feature Family | Plugin Name | Mathematical Representation | Scientific Use Case |
| :--- | :--- | :--- | :--- |
| **Cα Distances** | `protein.ca_distances_contacts` | $d_{ij} = \|\mathbf{r}_i - \mathbf{r}_j\|$ and $\Theta(d_{\text{cut}} - d_{ij})$ | Domain movements, hinge flexing |
| **Secondary Structure** | `protein.secondary_structure_dssp` | $\alpha$-helix, $\beta$-sheet, coil fractions | Unfolding, secondary structure transitions |
| **Backbone Dihedrals** | `protein.backbone_torsions` | $\sin(\phi), \cos(\phi), \sin(\psi), \cos(\psi)$ | Ramachandran regional shifts |
| **Sidechain Rotamers** | `protein.sidechain_torsions` | $\sin(\chi_1), \cos(\chi_1), \sin(\chi_2), \cos(\chi_2)$ | Hydrophobic core packing, allosteric triggers |
| **Solvent Exposure** | `solvation.sasa` | Surface area ($Å^2$) via Shrake-Rupley / FreeSASA | Cavity opening, hydrophobic burial |
| **Salt Bridges** | `protein.salt_bridges` | $d(\text{Arg/Lys}^+, \text{Asp/Glu}^-) \le 4.0$ Å | Electrostatic latching, pH stability |
| **Aromatic Stacking** | `protein.aromatic_stacking` | Centroid distance $d \le 5.5$ Å, normal angle $\theta$ | $\pi$-network stabilization |


## 3. Configuring and Extracting Multi-Family Panels


In [ ]:
from pathlib import Path
from molscope.api import Study

# Locate 2RJY dataset
data_candidates = [
    Path("../mdxplain study/mdxplain/data/2RJY"),
    Path("data/2RJY"),
    Path("/Users/nirav/Docs/mdxplain study/mdxplain/data/2RJY"),
]
data_dir = next((p for p in data_candidates if (p / "protein.pdb").is_file()), None)

if data_dir is not None:
    study = Study(project_id="feature_showcase", out_dir="results/features")
    study.add_system(
        system_id="villin",
        topology=data_dir / "protein.pdb",
        trajectories=[data_dir / "simulation_run1.xtc"],
    )
    study.configure_imaging(unwrap=True, center_selection="protein")
    study.configure_features([
        "protein.ca_distances_contacts",
        "protein.secondary_structure_dssp",
        "protein.sidechain_torsions",
    ])
    print("Multi-family panel configured.")


## 4. Feature Redundancy and Correlation Analysis

Extracting many features can introduce high collinearity. MolScope assesses pairwise Spearman rank correlations ($|r_s|$) to ensure selected features provide orthogonal biophysical information.


In [ ]:
print("To explore feature correlation maps and quality metrics:")
print("  study.get_quality_metrics()")
print("  study.get_features().corr(method='spearman')")
